In [1]:
import torch
import adabmDCA 
from adabmDCA.fasta import get_tokens, write_fasta
from adabmDCA.io import load_params, load_chains, import_from_fasta
from adabmDCA.utils import init_parameters, init_chains, get_device, get_dtype
from adabmDCA.functional import one_hot

import matplotlib.pyplot as plt
import numpy as np

from adabmDCA.statmech import compute_energy


# Set the device
device = get_device("cpu")
dtype = get_dtype("float32")

/home/robertonetti/Programs/miniconda3/envs/PLM/lib/python3.12/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Running on CPU


In [2]:
def compute_seqID(a1: torch.Tensor, single_seq: torch.Tensor):
    """
    Computes the Hamming distance 
    between a set of one-hot encoded sequences and a single one-hot encoded sequence.

    Args:
        a1 (torch.Tensor): Sequence dataset, shape (N, L, C), where N is the number of sequences,
                           L is the length, and C is the number of categories (one-hot size).
        single_seq (torch.Tensor): Single one-hot encoded sequence, shape (L, C).

    Returns:
        torch.Tensor: Hamming distances for each sequence in the dataset.
    """
    # print(a1.shape, single_seq.shape)
    a1 = a1.view(a1.shape[0], -1)
    single_seq = single_seq.view(1, -1)
    # print(a1.shape, single_seq.shape)
    seqID = (a1 * single_seq).sum(1) 

    return seqID

Extract Sequences and Model

Unknown token found: removing sequence gi|297170754|gb|ADI21776.1| hypothetical protein [uncultured gamma proteobacterium HF0130_22O14]


Test if seqID is computed well

In [10]:
print(compute_seqID(batch1, batch1[0]))
x = batch1[0] + batch1[248]
count = (x == 2).sum().item() 

tensor([96., 26., 22., 24., 22., 27., 32., 29., 30., 27., 24., 27., 22., 27.,
        23., 28., 28., 25., 29., 26., 29., 31., 29., 29., 30., 73., 28., 27.,
        23., 24., 29., 25., 23., 26., 27., 29., 31., 29., 25., 31., 25., 29.,
        24., 26., 28., 30., 26., 23., 28., 25., 27., 29., 27., 27., 29., 29.,
        29., 25., 27., 28., 27., 26., 28., 28., 30., 27., 28., 25., 28., 29.,
        25., 28., 29., 71., 28., 26., 26., 25., 31., 24., 23., 31., 29., 24.,
        27., 23., 26., 29., 26., 26., 28., 25., 24., 26., 26., 24., 28., 28.,
        28., 31., 30., 28., 29., 29., 29., 28., 26., 25., 27., 28., 30., 23.,
        28., 25., 23., 27., 25., 27., 29., 28., 27., 23., 27., 27., 37., 30.,
        24., 25., 24., 26., 26., 26., 26., 24., 27., 30., 25., 27., 27., 23.,
        27., 28., 26., 25., 28., 27., 29., 28., 67., 28., 26., 28., 26., 23.,
        28., 25., 25., 25., 26., 27., 24., 69., 66., 29., 30., 24., 27., 25.,
        24., 30., 29., 29., 27., 29., 23., 29., 29., 23., 26., 2

Re-compute seq DIVERGENCE

In [11]:
divergence1 = torch.zeros(M_sanple, device=device, dtype=dtype)
for i, sequence in enumerate(batch1):
        # Calcola il valore di divergence per l'indice corrente
        divergence1[i] = min(L - compute_seqID(nat_data, sequence)) / L
        

divergence2 = torch.zeros(M_sanple, device=device, dtype=dtype)
for i, sequence in enumerate(batch2):
        # Calcola il valore di divergence per l'indice corrente
        divergence2[i] = min(L - compute_seqID(nat_data, sequence)) / L


divergence3 = torch.zeros(M_sanple, device=device, dtype=dtype)
for i, sequence in enumerate(batch3):
        # Calcola il valore di divergence per l'indice corrente
        divergence3[i] = min(L - compute_seqID(nat_data, sequence)) / L

tensor([0.6042, 0.6250, 0.6354, 0.6146, 0.6146, 0.6042, 0.6146, 0.6042, 0.6042,
        0.6146, 0.6042, 0.6354, 0.6458, 0.6042, 0.6146, 0.6042, 0.6146, 0.6042,
        0.6042, 0.6042, 0.6042, 0.6042, 0.6146, 0.6042, 0.6042, 0.6042, 0.6146,
        0.6042, 0.6042, 0.6354, 0.6042, 0.6146, 0.6042, 0.6146, 0.6042, 0.6042,
        0.6042, 0.6146, 0.6042, 0.6146, 0.6146, 0.6042, 0.6146, 0.6042, 0.6146,
        0.6146, 0.6250, 0.6146, 0.6354, 0.6146, 0.6042, 0.6250, 0.6250, 0.6042,
        0.6042, 0.6250, 0.6146, 0.6458, 0.6146, 0.6354, 0.6250, 0.6146, 0.6042,
        0.6042, 0.6042, 0.6042, 0.6042, 0.6042, 0.6042, 0.6042, 0.6146, 0.6042,
        0.6250, 0.6042, 0.6042, 0.6146, 0.6042, 0.6042, 0.6146, 0.6042, 0.6042,
        0.6042, 0.6042, 0.6354, 0.6042, 0.6042, 0.6042, 0.6042, 0.6354, 0.6146,
        0.6146, 0.6042, 0.6354, 0.6146, 0.6146, 0.6042, 0.6146, 0.6042, 0.6146,
        0.6146, 0.6042, 0.6250, 0.6042, 0.6042, 0.6042, 0.6146, 0.6042, 0.6146,
        0.6146, 0.6042, 0.6042, 0.6354, 

In [30]:
count

31